# Parte 1: ETL - Verificação, Limpeza e Preparações dos Dados

Nesta etapa, focaremos em inspecionar os dados do **Brazilian E-Commerce Public Dataset by Olist**, provar através de análises (`.info`, `.isnull`) a necessidade das limpezas, tratar as variáveis e preparar nosso Dataset para receber Embeddings.

In [ ]:
import pandas as pd
import os
import re
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. Carregamento e Verificação dos Dados
Vamos carregar as tabelas principais envolvidas na Voz do Cliente e cruzar com produtos e pedidos. A verificação da qualidade dos dados nos dirá o que deve ser tratado.

In [ ]:
df_reviews = pd.read_csv(f'{DATA_DIR}olist_order_reviews_dataset.csv')
df_orders = pd.read_csv(f'{DATA_DIR}olist_orders_dataset.csv')
df_items = pd.read_csv(f'{DATA_DIR}olist_order_items_dataset.csv')
df_products = pd.read_csv(f'{DATA_DIR}olist_products_dataset.csv')

print("Visão geral de reviews:")
print(df_reviews.info())
print("\nValores ausentes em reviews:")
print(df_reviews.isnull().sum())

## 2. Tratamento de Valores Ausentes
Observamos acima que mais da metade das avaliações não possui um `review_comment_message`. 
**Justificativa:** Num sistema de RAG, apenas textos geram significado (vetores ricos). Uma linha sem texto de avaliação não nos diz os "motivos de insatisfação" ou "elogios", que são os requisitos do nosso desafio. Portanto, faremos a remoção explícita de registros nulos nessa coluna.

In [ ]:
total_antes = len(df_reviews)
df_reviews_text = df_reviews.dropna(subset=['review_comment_message']).copy()
total_depois = len(df_reviews_text)

print(f"Foram removidos {total_antes - total_depois} registros sem comentário (apenas com nota numéria).")
print(f"Registros restantes úteis para o RAG: {total_depois}")

## 3. Limpeza e Preparação dos Textos
Textos brutos vindos da web contém quebras de linha (`\n`, `\r`), múltiplos espaços e eventuais caracteres indesejados. Isso quebra tokens dos modelos de Embeddings.
**Justificativa:** Normalizar o espaçamento e manter tudo em uma linha contínua reduz o ruído para a similaridade de vetores.

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_reviews_text['review_comment_message'] = df_reviews_text['review_comment_message'].apply(clean_text)
# Preenche títulos vazios com espaço branco para evitar Nulos no join com string futuramente
df_reviews_text['review_comment_title'] = df_reviews_text['review_comment_title'].fillna('')

df_reviews_text[['review_comment_title', 'review_comment_message']].head()

## 4. Seleção de Dados e Consolidação (Join)
Precisamos de metadados como a categoria do produto (`product_category_name`). 
**Justificativa:** Sem a categoria, um modelo GenAI não consegue inferir de qual setor o cliente está falando, e não conseguiremos responder à pergunta "O que os clientes dizem sobre Perfumaria?".

In [ ]:
df_merged = df_reviews_text.merge(df_orders[['order_id', 'order_purchase_timestamp']], on='order_id', how='left')
df_merged = df_merged.merge(df_items[['order_id', 'product_id']], on='order_id', how='left')
df_merged = df_merged.merge(df_products[['product_id', 'product_category_name']], on='product_id', how='left')

print("Checando categorias nulas geradas pelo left join ou falta de cadastro do produto:")
print(df_merged['product_category_name'].isnull().sum())

# Tratamento de ausentes na categoria
df_merged['product_category_name'] = df_merged['product_category_name'].fillna('Desconhecida')

## 5. Organização dos Documentos
Alguns pedidos possuem múltiplos itens de categorias diferentes. Para evitar avaliações duplicadas inflando nosso Vector DB, unimos categorias de um mesmo ID de avaliação (`review_id`). E por fim, formataremos isso de forma legível em uma string.

In [ ]:
df_final = df_merged.groupby('review_id').agg({
    'review_score': 'first',
    'review_comment_title': 'first',
    'review_comment_message': 'first',
    'order_purchase_timestamp': 'first',
    'product_category_name': lambda x: ", ".join(list(set([str(i) for i in x])))
}).reset_index()

def criar_documento(row):
    titulo = f" Título: {row['review_comment_title']} |" if row['review_comment_title'] else ""
    return f"Categoria: {row['product_category_name']} | Nota: {row['review_score']} | Data: {row['order_purchase_timestamp']} |{titulo} Comentário: {row['review_comment_message']}"

df_final['rag_document'] = df_final.apply(criar_documento, axis=1)

print("Exemplo do Documento final que será inserido na base de Embeddings:")
print(df_final['rag_document'].iloc[0])

df_final[['review_id', 'rag_document', 'review_score']].to_csv(f'{PROCESSED_DIR}reviews_enriched_for_rag.csv', index=False)
print("\nDocumento salvo e preparado para a Fase de Embeddings!")